# P2 — TaxPilot Document Extractor

**Role:** Extraction Engineer  
**Topics:** LangChain, AI Agent  
**Output:** `TaxpayerProfile` JSON + AIS vs Declaration mismatch list → feeds into P3 LangGraph `run_extractor` node

---
### Architecture of this notebook

```
┌─────────────────────────────────────────────────────────┐
│  SECTION 1: pip installations                           │
│  SECTION 2: imports                                     │
│  SECTION 3: Pydantic models  ◄── LangChain schema       │
│  SECTION 4: raw document strings (Form 16, AIS, proofs) │
│  SECTION 5: LangChain extractor chain                   │
│             llm.with_structured_output(TaxpayerProfile) │
│  SECTION 6: run extraction (invoke the chain)           │
│  SECTION 7: AIS vs Declaration diff (mismatch detect)   │
│  SECTION 8: extractor_node() → P3 LangGraph node func   │
│  SECTION 9: final output in P3's expected format        │
└─────────────────────────────────────────────────────────┘
```

> **No MCP here** — MCP server is P1's job; MCP *consumer* is P4's job.  
> **No LangGraph graph here** — the graph + interrupt gates are P3's job.  
> This notebook produces the typed `taxpayer_profile` dict that P3 reads.

---
## SECTION 1 — Installations

Run once. Restart the kernel after this cell completes.

In [ ]:
# ── INSTALLATIONS ────────────────────────────────────────────
# langchain-anthropic  → ChatAnthropic model
# langchain            → ChatPromptTemplate, with_structured_output
# pydantic             → data models for structured extraction
# python-dotenv        → load ANTHROPIC_API_KEY from .env

!pip install langchain langchain-anthropic pydantic python-dotenv --quiet

---
## SECTION 2 — Imports

All imports in one place so Jupyter can be re-run cleanly.

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────

# Standard library
import os
import json
from typing import Any, Optional
from dataclasses import dataclass, asdict

# Env
from dotenv import load_dotenv

# ── LangChain imports ─────────────────────────────────────────
from langchain_anthropic import ChatAnthropic          # chat model wrapper
from langchain_core.prompts import ChatPromptTemplate  # prompt builder

# ── Pydantic (used by LangChain with_structured_output) ───────
from pydantic import BaseModel, Field

load_dotenv()  # reads ANTHROPIC_API_KEY from .env file
print("Imports OK")

---
## SECTION 3 — Pydantic Models (LangChain Structured Output Schema)

**What:** Pydantic models define the exact JSON shape that LangChain's
`with_structured_output()` forces the Claude model to return.

**Why here and not in LangGraph:** these are *data contracts*, not graph nodes.
They are shared across P2 (extraction) and P3 (state typing).

In [ ]:
# ── PYDANTIC MODELS ───────────────────────────────────────────
# LangChain calls these as a JSON schema when it binds to the model.

class Form16Data(BaseModel):
    """Employer-issued salary certificate."""
    employer: str
    gross_salary: float
    basic: float
    hra_received: float
    professional_tax: float
    tds_deducted: float


class AISData(BaseModel):
    """Annual Information Statement from the income-tax dept."""
    salary: float
    savings_interest: float
    fd_interest: float            # ← deliberate trap: 38 000 not declared
    tds_total: float


class Section80C(BaseModel):
    epf: float = 0
    elss: float = 0
    ppf: float = 0
    life_insurance: float = 0

    @property
    def total(self) -> float:
        return self.epf + self.elss + self.ppf + self.life_insurance


class DeclaredDeductions(BaseModel):
    """What the taxpayer told us they have."""
    rent_paid_annual: float = 0
    rent_receipts_available: bool = False
    sec_80c: Section80C = Field(default_factory=Section80C)
    sec_80d_self: float = 0
    savings_interest: float = 0   # taxpayer-declared savings interest


class TaxpayerProfile(BaseModel):
    """Root model — the complete structured output from the extractor."""
    name: str
    age: int
    city: str
    is_metro: bool
    fy: str                       # e.g. "2025-26"
    form16: Form16Data
    ais: AISData
    declared: DeclaredDeductions


print("Pydantic models defined:")
print("  TaxpayerProfile → Form16Data, AISData, DeclaredDeductions, Section80C")

---
## SECTION 4 — Raw Document Strings

In production these would be PDF/image text extracted by an OCR step.
For this exercise we use the sample data from the problem statement.

In [ ]:
# ── RAW DOCUMENTS ─────────────────────────────────────────────
# Simulates text extracted from scanned PDFs.

RAW_FORM16 = """
FORM 16 — FY 2025-26
Employer     : Acme Software Pvt Ltd
Employee     : Priya Sharma
Gross Salary : Rs 18,00,000
Basic Salary : Rs 7,20,000
HRA Received : Rs 2,88,000
Professional Tax : Rs 2,400
TDS Deducted : Rs 1,95,000
"""

RAW_AIS = """
ANNUAL INFORMATION STATEMENT (AIS) — FY 2025-26
Taxpayer : Priya Sharma

Income Sources:
  Salary Income               : Rs 18,00,000  (Acme Software Pvt Ltd)
  Savings Account Interest    : Rs 12,000
  Fixed Deposit Interest      : Rs 38,000      ← NOT declared by taxpayer
  TDS Deducted (Total)        : Rs 1,95,000
"""

RAW_PROOFS = """
TAXPAYER DECLARATION & INVESTMENT PROOFS
Name   : Priya Sharma
Age    : 31 years
City   : Bengaluru  (metro)
FY     : 2025-26

Housing:
  Annual Rent Paid            : Rs 3,00,000
  Rent Receipts Available     : Yes

Section 80C Investments:
  EPF Contribution            : Rs 86,400
  ELSS Mutual Fund            : Rs 63,600

Section 80D — Health Insurance:
  Self & Family Premium       : Rs 25,000

Interest Income:
  Savings Account Interest    : Rs 12,000
  (No FD interest declared)
"""

print("Raw documents ready: Form 16, AIS, Taxpayer Proofs")

---
## SECTION 5 — LangChain Extractor Chain

**Core P2 component.** Three steps:
1. `ChatAnthropic(...)` — the Claude model
2. `.with_structured_output(TaxpayerProfile)` — forces JSON matching our Pydantic schema
3. `ChatPromptTemplate | structured_llm` — the chain

> `with_structured_output` uses tool-calling under the hood:
> Claude is given the Pydantic JSON schema as a tool definition and
> *must* call that tool, guaranteeing a valid, typed response.

In [ ]:
# ── LANGCHAIN EXTRACTOR CHAIN ─────────────────────────────────

# Step 1: base LLM
llm = ChatAnthropic(
    model="claude-sonnet-4-5",
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    temperature=0,   # deterministic extraction
)

# Step 2: bind structured output → forces Claude to return TaxpayerProfile JSON
structured_llm = llm.with_structured_output(TaxpayerProfile)

# Step 3: prompt template
extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an Indian income-tax extraction specialist.
Parse Form 16, AIS, and taxpayer proofs into the required JSON.
Rules:
- Convert all Indian number formats (Rs X,XX,XXX) to plain floats.
- Metro cities: Mumbai, Delhi, Kolkata, Chennai, Bengaluru, Hyderabad → is_metro=true.
- Only populate declared.savings_interest from the taxpayer's own declaration,
  not from AIS (AIS FD interest belongs in ais.fd_interest only).
"""),
    ("human", """Extract the taxpayer profile from these three documents:

=== FORM 16 ===
{form16_text}

=== ANNUAL INFORMATION STATEMENT (AIS) ===
{ais_text}

=== TAXPAYER DECLARATIONS & PROOFS ===
{proofs_text}

Return a complete TaxpayerProfile with all fields."""),
])

# Step 4: compose the chain  (prompt → structured_llm)
extractor_chain = extraction_prompt | structured_llm

print("LangChain extractor chain built:")
print("  ChatPromptTemplate | ChatAnthropic.with_structured_output(TaxpayerProfile)")

---
## SECTION 6 — Run Extraction

Invoke the chain with the raw document strings. Claude returns a
fully-validated `TaxpayerProfile` Pydantic object — no JSON parsing needed.

In [ ]:
# ── RUN EXTRACTION ────────────────────────────────────────────

print("Calling Claude via LangChain...")

profile: TaxpayerProfile = extractor_chain.invoke({
    "form16_text": RAW_FORM16,
    "ais_text":    RAW_AIS,
    "proofs_text": RAW_PROOFS,
})

print("\n✅ Extraction complete!")
print(f"   Taxpayer : {profile.name}, age {profile.age}, {profile.city} (metro={profile.is_metro})")
print(f"   FY       : {profile.fy}")
print()
print("── Form 16 ─────────────────────────────────────")
print(f"   Gross salary  : Rs {profile.form16.gross_salary:>12,.0f}")
print(f"   Basic         : Rs {profile.form16.basic:>12,.0f}")
print(f"   HRA received  : Rs {profile.form16.hra_received:>12,.0f}")
print(f"   Prof. tax     : Rs {profile.form16.professional_tax:>12,.0f}")
print(f"   TDS           : Rs {profile.form16.tds_deducted:>12,.0f}")
print()
print("── AIS ─────────────────────────────────────────")
print(f"   Salary        : Rs {profile.ais.salary:>12,.0f}")
print(f"   Savings int.  : Rs {profile.ais.savings_interest:>12,.0f}")
print(f"   FD interest   : Rs {profile.ais.fd_interest:>12,.0f}  ← trap")
print(f"   TDS total     : Rs {profile.ais.tds_total:>12,.0f}")
print()
print("── Declared ────────────────────────────────────")
print(f"   Rent paid     : Rs {profile.declared.rent_paid_annual:>12,.0f}")
print(f"   Receipts      : {profile.declared.rent_receipts_available}")
print(f"   80C (EPF)     : Rs {profile.declared.sec_80c.epf:>12,.0f}")
print(f"   80C (ELSS)    : Rs {profile.declared.sec_80c.elss:>12,.0f}")
print(f"   80D self      : Rs {profile.declared.sec_80d_self:>12,.0f}")
print(f"   Savings int.  : Rs {profile.declared.savings_interest:>12,.0f}")

---
## SECTION 7 — AIS vs Declaration Diff (Mismatch Detection)

Compares income reported by the tax department (AIS) with what the
taxpayer declared. Any gap is a potential mismatch that must be
escalated to HITL Gate 1 in P3.

**Expected finding:** FD interest of Rs 38,000 in AIS but Rs 0 declared.

In [ ]:
# ── AIS vs DECLARATION DIFF ───────────────────────────────────

@dataclass
class MismatchItem:
    id: str           # stable key used by P3 Gate 1 decisions
    type: str         # human-readable category
    amount: float     # AIS amount
    declared: bool    # was it declared by the taxpayer?
    source: str       # "AIS"


def compute_ais_diff(profile: TaxpayerProfile) -> list[MismatchItem]:
    """
    Returns a list of income items in AIS that differ from taxpayer's declaration.
    Each item becomes one decision for HITL Gate 1.
    """
    mismatches: list[MismatchItem] = []

    # ── Check 1: FD interest ──────────────────────────────────
    # AIS reports fd_interest; taxpayer declared nothing for FD.
    # declared.savings_interest covers only savings accounts.
    if profile.ais.fd_interest > 0:
        # Taxpayer has no field for FD interest → default 0
        declared_fd = 0
        if abs(profile.ais.fd_interest - declared_fd) > 1:
            mismatches.append(MismatchItem(
                id="AIS-FD-001",
                type="FD interest",
                amount=profile.ais.fd_interest,
                declared=False,
                source="AIS",
            ))

    # ── Check 2: Savings account interest ────────────────────
    if abs(profile.ais.savings_interest - profile.declared.savings_interest) > 1:
        mismatches.append(MismatchItem(
            id="AIS-SAV-001",
            type="Savings interest",
            amount=profile.ais.savings_interest,
            declared=False,
            source="AIS",
        ))

    # ── Check 3: Salary match between AIS and Form 16 ────────
    if abs(profile.ais.salary - profile.form16.gross_salary) > 1:
        mismatches.append(MismatchItem(
            id="AIS-SAL-001",
            type="Salary mismatch (AIS vs Form 16)",
            amount=profile.ais.salary,
            declared=False,
            source="AIS",
        ))

    return mismatches


# Run diff
mismatches = compute_ais_diff(profile)

print("━" * 55)
print("  AIS vs DECLARATION — MISMATCH REPORT")
print("━" * 55)
if not mismatches:
    print("  ✅ No mismatches found.")
else:
    print(f"  ⚠️  {len(mismatches)} mismatch(es) found — will trigger HITL Gate 1:\n")
    for m in mismatches:
        status = "NOT declared" if not m.declared else "declared"
        print(f"  [{m.id}]  {m.type}")
        print(f"           Source : {m.source}")
        print(f"           Amount : Rs {m.amount:,.0f}")
        print(f"           Status : {status}")
        print()

---
## SECTION 8 — Extractor Node (LangGraph Integration Point)

This function is the **LangGraph node** that P3 will call as `"extractor"`.
P3's `orchestrator.py` already has a stub:
```python
def run_extractor(state: TaxState) -> dict[str, Any]:
    profile = state.get("taxpayer_profile")   # ← P2 must populate this
    ...
```

P2 produces `taxpayer_profile` *before* the graph starts (pre-filled into
`TaxState`), **or** P3 can call `extractor_node()` directly as the first node.

> **Node vs Chain:**  
> - *Chain* = the LangChain pipeline (`extraction_prompt | structured_llm`)  
> - *Node* = the Python function registered with `graph.add_node()`  
> The node wraps the chain and adds the diff logic.

In [ ]:
# ── EXTRACTOR NODE FUNCTION ───────────────────────────────────
# This is what P3 would call: graph.add_node("extractor", extractor_node)

def extractor_node(state: dict) -> dict:
    """
    LangGraph node — runs document extraction + AIS diff.

    Reads from state:
        form16_text  (str) : raw Form 16 text
        ais_text     (str) : raw AIS text
        proofs_text  (str) : raw taxpayer proof text

    Writes to state:
        taxpayer_profile  (dict)       : structured profile + income_heads
        mismatches        (list[dict]) : items for Gate 1
        status            (str)        : "extracted"
    """
    # ── Step A: LangChain extraction ──────────────────────────
    raw_profile: TaxpayerProfile = extractor_chain.invoke({
        "form16_text": state.get("form16_text", RAW_FORM16),
        "ais_text":    state.get("ais_text",    RAW_AIS),
        "proofs_text": state.get("proofs_text", RAW_PROOFS),
    })

    # ── Step B: AIS diff ──────────────────────────────────────
    mismatch_items = compute_ais_diff(raw_profile)

    # ── Step C: Build income_heads (all AIS income) ───────────
    income_heads = {
        "salary":           raw_profile.ais.salary,
        "savings_interest": raw_profile.ais.savings_interest,
        "fd_interest":      raw_profile.ais.fd_interest,
    }

    # ── Step D: Compose taxpayer_profile dict for TaxState ────
    taxpayer_profile = {
        **raw_profile.model_dump(),
        "income_heads": income_heads,
        "mismatches":   [asdict(m) for m in mismatch_items],
    }

    return {
        "taxpayer_profile": taxpayer_profile,
        "mismatches":       [asdict(m) for m in mismatch_items],
        "status":           "extracted",
    }


print("extractor_node() defined — ready for P3 graph.add_node('extractor', extractor_node)")

---
## SECTION 9 — Final Output in P3's Expected Format

Test the full node end-to-end and print the exact dict structure that
`orchestrator.py`'s `TaxState` consumes.

In [ ]:
# ── FULL END-TO-END TEST ──────────────────────────────────────

print("Running extractor_node() end-to-end...")
output = extractor_node({})

print("\n✅ extractor_node() output (P3-ready):\n")

# Summarise key fields
prof = output["taxpayer_profile"]
print(f"  taxpayer_profile.name          : {prof['name']}")
print(f"  income_heads.salary            : Rs {prof['income_heads']['salary']:>12,.0f}")
print(f"  income_heads.savings_interest  : Rs {prof['income_heads']['savings_interest']:>12,.0f}")
print(f"  income_heads.fd_interest       : Rs {prof['income_heads']['fd_interest']:>12,.0f}  ← mismatch")
print(f"  mismatches count               : {len(output['mismatches'])}")
print(f"  status                         : {output['status']}")
print()

print("── Mismatches (feeds HITL Gate 1) ──────────────")
for m in output["mismatches"]:
    print(f"  {m['id']}  {m['type']}  Rs {m['amount']:,.0f}  declared={m['declared']}")

print()
print("── Full taxpayer_profile JSON ───────────────────")
print(json.dumps(output["taxpayer_profile"], indent=2, ensure_ascii=False))

---
## SECTION 10 — How to Plug into P3's orchestrator.py

P3 already has a stub `run_extractor` node. Replace the stub body,
or pre-populate `TaxState` before invoking the graph:

```python
# Option A — replace P3's stub with the real node
# In orchestrator.py, change:
#
#   def run_extractor(state: TaxState) -> dict[str, Any]:
#       profile = state.get("taxpayer_profile")
#       ...
#
# To:
#   from P2_extractor import extractor_node as run_extractor

# Option B — pre-populate TaxState (simpler for hackathon demo)
from P2_extractor import extractor_node

extracted = extractor_node({})

initial_state = {
    "taxpayer_id":      "PRIYA-001",
    "taxpayer_profile": extracted["taxpayer_profile"],   # ← P2 output
    "regime_comparison": { ... },                        # ← P4 output
}
result = app.invoke(initial_state, config=config)
```

---
### What each section covers

| Section | Component | Topic |
|---------|-----------|-------|
| 1 | `pip install` | Installations |
| 2 | `import` block | Imports |
| 3 | Pydantic models | LangChain (schema for structured output) |
| 4 | Raw document strings | Sample data |
| 5 | `extractor_chain` | **LangChain** — `with_structured_output` |
| 6 | `extractor_chain.invoke()` | LangChain chain run |
| 7 | `compute_ais_diff()` | Mismatch detection logic |
| 8 | `extractor_node()` | **Node** — callable for `graph.add_node()` |
| 9 | End-to-end test | Validates expected output |
| 10 | Integration guide | How P3 consumes P2's output |

> **Not in this notebook (by design):**
> - MCP server or tool calls → P1 / P4
> - `StateGraph`, `interrupt()`, `Command` → P3
> - Langfuse callbacks → P5